# Buffalo + Liveness Video Authentication Pipeline

Pipeline order:
1. Input video
2. Buffalo detect (det_10g)
3. Face alignment (2d106det / keypoint-based alignment)
4. Liveness ViT
5. Random frame sampling
6. Buffalo embedding (w600k_r50 via InsightFace recognition embedding)
7. Similarity comparison
8. Authenticate / Reject

This notebook assumes registration will be added later. For now, you can use a folder of reference images as the enrolled identity.

In [ ]:
from pathlib import Path
from datetime import datetime
import random
import json

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from tqdm.auto import tqdm
from insightface.app import FaceAnalysis
from insightface.utils import face_align

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("device:", device)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# -----------------------------
# Config
# -----------------------------
VIDEO_PATH = Path(r"C:\DSP\auth_video.mp4")
REFERENCE_IMAGE_DIR = Path(r"C:\DSP\reference_images")
OUTPUT_ROOT = Path(r"C:\DSP\buffalo_pipeline_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR = RUN_DIR / "crops"
CROPS_DIR.mkdir(exist_ok=True)

LIVENESS_CHECKPOINT_PATH = Path(r"C:\DSP\face_liveness_vit\model.pt")
INSIGHTFACE_MODEL_NAME = "buffalo_l"
INSIGHTFACE_ROOT = None
DET_SIZE = (640, 640)
FACE_SIZE = 224
MAX_SCAN_FACES = 48
MAX_SAMPLED_FACES = 12
MIN_FACES_REQUIRED = 6
RANDOM_SEED = 42

LIVENESS_SPOOF_THRESHOLD = 0.1199
MIN_LIVE_RATIO = 0.60
MATCH_THRESHOLD = 0.35
MIN_MATCH_RATIO = 0.50
USE_MEDIAN_SCORE = True

assert LIVENESS_CHECKPOINT_PATH.exists(), LIVENESS_CHECKPOINT_PATH
print("video:", VIDEO_PATH)
print("reference dir:", REFERENCE_IMAGE_DIR)
print("run dir:", RUN_DIR)
print("liveness checkpoint:", LIVENESS_CHECKPOINT_PATH)

In [ ]:
# -----------------------------
# Liveness ViT model
# -----------------------------
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=int(d_model * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        return self.transformer_encoder(x)


class LivenessViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        d_model=128,
        nhead=4,
        num_layers=2,
        num_classes=2,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=3, embed_dim=d_model)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, d_model))
        self.pos_drop = nn.Dropout(dropout)
        self.encoder = Encoder(d_model=d_model, nhead=nhead, num_layers=num_layers, mlp_ratio=mlp_ratio, dropout=dropout)
        self.head = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        batch_size = x.size(0)
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.pos_drop(x)
        x = self.encoder(x)
        cls_out = x[:, 0]
        return self.head(cls_out)

In [ ]:
# -----------------------------
# Load Buffalo + Liveness
# -----------------------------
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device.type == "cuda" else ['CPUExecutionProvider']
face_kwargs = {"name": INSIGHTFACE_MODEL_NAME, "providers": providers}
if INSIGHTFACE_ROOT:
    face_kwargs["root"] = INSIGHTFACE_ROOT

face_app = FaceAnalysis(**face_kwargs)
face_app.prepare(ctx_id=0 if device.type == "cuda" else -1, det_size=DET_SIZE)

liveness_model = LivenessViT(
    img_size=224,
    patch_size=16,
    d_model=128,
    nhead=4,
    num_layers=2,
    num_classes=2,
).to(device)

liveness_checkpoint = torch.load(LIVENESS_CHECKPOINT_PATH, map_location=device, weights_only=False)
liveness_state = liveness_checkpoint["model"] if isinstance(liveness_checkpoint, dict) and "model" in liveness_checkpoint else liveness_checkpoint
liveness_model.load_state_dict(liveness_state, strict=True)
liveness_model.eval()

liveness_transform = transforms.Compose([
    transforms.Resize(int(FACE_SIZE * 1.14)),
    transforms.CenterCrop(FACE_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Buffalo loaded")
print("Liveness model loaded")

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def pick_largest_face(faces):
    if not faces:
        return None
    return sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)[0]


def get_aligned_face_224(frame_bgr, face_obj, out_size=224):
    if hasattr(face_obj, "kps") and face_obj.kps is not None:
        aligned_bgr = face_align.norm_crop(frame_bgr, landmark=face_obj.kps, image_size=out_size)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(aligned_rgb)

    x1, y1, x2, y2 = face_obj.bbox.astype(int)
    h, w = frame_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_bgr = cv2.resize(crop_bgr, (out_size, out_size))
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


def image_paths_from_dir(folder):
    if not folder.exists():
        raise FileNotFoundError(folder)
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])


def load_reference_embeddings(reference_dir):
    image_paths = image_paths_from_dir(reference_dir)
    if len(image_paths) == 0:
        raise RuntimeError(f"No reference images found in {reference_dir}")

    embeddings = []
    used_paths = []
    progress = tqdm(image_paths, desc="reference embeddings", dynamic_ncols=True)
    for image_path in progress:
        frame_bgr = cv2.imread(str(image_path))
        if frame_bgr is None:
            continue
        faces = face_app.get(frame_bgr)
        face = pick_largest_face(faces)
        if face is None:
            continue
        emb = np.asarray(face.embedding, dtype=np.float32)
        emb = emb / np.linalg.norm(emb)
        embeddings.append(emb)
        used_paths.append(str(image_path))
        progress.set_postfix({"kept": len(embeddings)})

    if len(embeddings) == 0:
        raise RuntimeError("No usable reference faces found")

    embeddings = np.stack(embeddings, axis=0)
    mean_embedding = embeddings.mean(axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    return {
        "used_paths": used_paths,
        "embeddings": embeddings,
        "mean_embedding": mean_embedding,
    }


def collect_video_faces(video_path):
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    collected = []
    frame_idx = 0
    progress = tqdm(desc="scan video", dynamic_ncols=True)
    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            faces = face_app.get(frame_bgr)
            face = pick_largest_face(faces)
            if face is not None:
                aligned = get_aligned_face_224(frame_bgr, face, out_size=224)
                if aligned is not None:
                    crop_path = CROPS_DIR / f"frame_{frame_idx:06d}.jpg"
                    aligned.save(crop_path, quality=95)
                    embedding = np.asarray(face.embedding, dtype=np.float32)
                    embedding = embedding / np.linalg.norm(embedding)
                    collected.append({
                        "frame_index": frame_idx,
                        "aligned_face": aligned,
                        "embedding": embedding,
                        "crop_path": str(crop_path),
                    })

            progress.update(1)
            progress.set_postfix({"faces": len(collected), "frame": frame_idx})
            if len(collected) >= MAX_SCAN_FACES:
                break
            frame_idx += 1
    finally:
        cap.release()
        progress.close()

    return collected


def run_liveness_on_records(records):
    spoof_probs = []
    live_flags = []
    progress = tqdm(records, desc="liveness", dynamic_ncols=True)
    for record in progress:
        x = liveness_transform(record["aligned_face"]).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = liveness_model(x)
            probs = torch.softmax(logits, dim=1)
        spoof_prob = probs[0, 1].item()
        is_live = spoof_prob < LIVENESS_SPOOF_THRESHOLD
        record["spoof_prob"] = spoof_prob
        record["is_live"] = is_live
        spoof_probs.append(spoof_prob)
        live_flags.append(is_live)
        progress.set_postfix({"spoof": f"{spoof_prob:.4f}", "live": is_live})

    mean_spoof = sum(spoof_probs) / len(spoof_probs)
    live_ratio = sum(live_flags) / len(live_flags)
    session_live = (mean_spoof < LIVENESS_SPOOF_THRESHOLD) and (live_ratio >= MIN_LIVE_RATIO)
    return {
        "mean_spoof": mean_spoof,
        "live_ratio": live_ratio,
        "session_live": session_live,
    }


def sample_records(records, max_samples, seed=42):
    if len(records) < MIN_FACES_REQUIRED:
        raise RuntimeError(f"Need at least {MIN_FACES_REQUIRED} usable face crops, found {len(records)}")
    rng = random.Random(seed)
    sample_count = min(max_samples, len(records))
    return rng.sample(records, sample_count)


def compare_embeddings(sampled_records, reference_profile):
    ref_embeddings = reference_profile["embeddings"]
    ref_mean = reference_profile["mean_embedding"]

    per_frame_best = []
    for record in sampled_records:
        emb = record["embedding"]
        gallery_scores = ref_embeddings @ emb
        mean_score = float(np.dot(ref_mean, emb))
        best_score = max(float(gallery_scores.max()), mean_score)
        per_frame_best.append(best_score)
        record["similarity"] = best_score
        record["matched"] = best_score >= MATCH_THRESHOLD

    per_frame_best = np.asarray(per_frame_best, dtype=np.float32)
    session_score = float(np.median(per_frame_best) if USE_MEDIAN_SCORE else np.mean(per_frame_best))
    match_ratio = float(np.mean(per_frame_best >= MATCH_THRESHOLD))
    return {
        "session_score": session_score,
        "match_ratio": match_ratio,
        "authenticated": session_score >= MATCH_THRESHOLD and match_ratio >= MIN_MATCH_RATIO,
        "per_frame_scores": per_frame_best.tolist(),
    }

In [ ]:
# -----------------------------
# Load enrolled identity from reference images
# -----------------------------
reference_profile = load_reference_embeddings(REFERENCE_IMAGE_DIR)
print("reference images used:", len(reference_profile["used_paths"]))
for path in reference_profile["used_paths"]:
    print(" -", path)

In [ ]:
# -----------------------------
# Run pipeline on the input video
# -----------------------------
video_records = collect_video_faces(VIDEO_PATH)
print("detected/aligned faces:", len(video_records))
if len(video_records) < MIN_FACES_REQUIRED:
    raise RuntimeError(f"Need at least {MIN_FACES_REQUIRED} usable faces, got {len(video_records)}")

liveness_summary = run_liveness_on_records(video_records)
print("liveness summary:", liveness_summary)

live_records = [record for record in video_records if record["is_live"]]
usable_records = live_records if len(live_records) >= MIN_FACES_REQUIRED else video_records
sampled_records = sample_records(usable_records, MAX_SAMPLED_FACES, seed=RANDOM_SEED)
print("sampled records:", len(sampled_records))

comparison = compare_embeddings(sampled_records, reference_profile)
authenticated = bool(liveness_summary["session_live"] and comparison["authenticated"])

print("\n===== FINAL DECISION =====")
print("session live      :", liveness_summary["session_live"])
print("session similarity:", f"{comparison['session_score']:.4f}")
print("match ratio       :", f"{comparison['match_ratio']:.2%}")
print("authenticated     :", authenticated)

In [ ]:
# -----------------------------
# Per-frame details + save summary
# -----------------------------
frame_rows = []
for record in sampled_records:
    frame_rows.append({
        "frame_index": int(record["frame_index"]),
        "crop_path": record["crop_path"],
        "spoof_prob": float(record["spoof_prob"]),
        "is_live": bool(record["is_live"]),
        "similarity": float(record["similarity"]),
        "matched": bool(record["matched"]),
    })

for row in frame_rows:
    print(
        f"frame={row['frame_index']:05d} | "
        f"spoof={row['spoof_prob']:.4f} | "
        f"live={row['is_live']} | "
        f"sim={row['similarity']:.4f} | "
        f"matched={row['matched']}"
    )

summary = {
    "video_path": str(VIDEO_PATH),
    "reference_image_dir": str(REFERENCE_IMAGE_DIR),
    "detected_faces": len(video_records),
    "sampled_faces": len(sampled_records),
    "liveness": liveness_summary,
    "comparison": comparison,
    "authenticated": authenticated,
    "frame_rows": frame_rows,
}

summary_path = RUN_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("summary saved to:", summary_path)